# **DOWNLOAD DO CORPUS**
**Nome:** Gisela Kassick e Lívia Aragão

## Importações

In [ ]:
import requests
import pandas as pd
import time
import re

## Configurações

In [ ]:
API_KEY = "SEU_TOKEN"
OUTPUT = "Eskin_detalhado.xlsx"
MAX_RESULTS = 10000
PER_PAGE = 200

SEARCH = '("tactile sensor" OR "tactile sensing" OR "electronic skin" OR "e-skin" OR "skin-like electronics" OR "artificial sensory skin" OR "touch sensor" OR "touch sensing" OR "haptic sensor" OR "haptic sensing") AND (pressur* OR conduct* OR resist* OR capacitive* OR piezoresist* OR piezoelectric* OR triboelectric* OR strain* OR deform* OR mechanical* OR impedance* OR "force sensing" OR "force sensor")'


## Funções auxiliares

In [ ]:
def limpar_texto(texto):
    if not texto:
        return ""
    texto = str(texto)
    # Remove caracteres inválidos para o Excel
    texto = re.sub(r'[\x00-\x08\x0B\x0C\x0E-\x1F]', '', texto)
    return texto[:32767].strip()

def reconstruir_abstract(abstract_index):
    if not abstract_index:
        return ""
    palavras = []
    for palavra, posicoes in abstract_index.items():
        for pos in posicoes:
            palavras.append((int(pos), palavra))
    palavras.sort(key=lambda x: x[0])
    return limpar_texto(" ".join(palavra for _, palavra in palavras))

## Busca no OpenAlex

In [ ]:
url = "https://api.openalex.org/works"
params = {
    # search restrito ao campo abstract
    "filter": f"title_and_abstract.search.exact:{SEARCH},type:article,has_abstract:true",
    "sort": "relevance_score:desc",
    "per_page": PER_PAGE,
    "cursor": "*",
    "api_key": API_KEY,
}

dados = []
ids_processados = set()

print("Iniciando a busca no OpenAlex...")

while len(dados) < MAX_RESULTS:
    try:
        r = requests.get(url, params=params, timeout=60)
        if r.status_code != 200:
            print(f"Erro na API ({r.status_code}): {r.text}")
            break

        resposta = r.json()
        resultados = resposta.get("results", [])

        if not dados and not ids_processados:
            total_api = resposta.get("meta", {}).get("count")
            print(f"Total reportado pela API para esta busca+filtros: {total_api}")

        if not resultados:
            print("Não há mais resultados disponíveis para esta busca.")
            break

        for work in resultados:
            if len(dados) >= MAX_RESULTS:
                break

            openalex_id = work.get("id")
            if openalex_id in ids_processados:
                continue

            # Reconstrução e validação do abstract
            
            abstract_index = work.get("abstract_inverted_index")
            
            if not abstract_index:
                continue
            
            abstract = reconstruir_abstract(abstract_index)
            
            if not abstract:
                continue
            
            # Descarta abstracts muito curtos, provavelmente incompletos/truncados
            if len(abstract.split()) < 50:
                continue

            # Retira a palavra "Abstract" do início, se presente
            if abstract.lower().startswith("abstract"):
                abstract = abstract[len("abstract"):].lstrip(" :-.\n")
                
            ids_processados.add(openalex_id)

            # Extração de Autores
            autores = "; ".join([
                a.get("author", {}).get("display_name", "")
                for a in work.get("authorships", [])
                if a.get("author", {}).get("display_name")
            ])

            # Revista
            primary_loc = work.get("primary_location") or {}
            source = primary_loc.get("source") or {}
            revista = source.get("display_name", "")

            # Links PDF e Landing Page
            best_oa = work.get("best_oa_location") or {}
            pdf_url = best_oa.get("pdf_url", "")
            landing_url = best_oa.get("landing_page_url", "")

            dados.append({
                "Title": limpar_texto(work.get("title", "")),
                "Year": work.get("publication_year"),
                "Authors": limpar_texto(autores),
                "Journal": limpar_texto(revista),
                "DOI": limpar_texto(work.get("doi", "")),
                "Abstract": abstract,
                "Citations": work.get("cited_by_count"),
                "Type": work.get("type"),
                "Language": work.get("language"),
                "Open Access": work.get("open_access", {}).get("is_oa"),
                "PDF": limpar_texto(pdf_url),
                "Landing Page": limpar_texto(landing_url),
                "OpenAlex ID": limpar_texto(openalex_id),
            })

        print(f"Artigos coletados até agora: {len(dados)}")

        # Avança para a próxima página
        cursor = resposta.get("meta", {}).get("next_cursor")
        if not cursor:
            print("Fim das páginas de resultados.")
            break

        params["cursor"] = cursor
        time.sleep(0.1)

    except Exception as e:
        print(f"Erro durante a execução: {e}")
        break

## Salvar excel

In [1]:
if dados:
    df = pd.DataFrame(dados)
    df.to_excel(OUTPUT, index=False, engine="openpyxl")
    print()
    print(f"PROCESSO CONCLUÍDO!")
    print(f"Total de artigos baixados: {len(df):,}")
    print(f"Arquivo salvo em: {OUTPUT}")
else:
    print("Nenhum artigo foi encontrado ou baixado.")

Iniciando a busca no OpenAlex...
Total reportado pela API para esta busca+filtros: 5777
Artigos coletados até agora: 191
Artigos coletados até agora: 388
Artigos coletados até agora: 583
Artigos coletados até agora: 776
Artigos coletados até agora: 974
Artigos coletados até agora: 1165
Artigos coletados até agora: 1361
Artigos coletados até agora: 1555
Artigos coletados até agora: 1749
Artigos coletados até agora: 1946
Artigos coletados até agora: 2141
Artigos coletados até agora: 2334
Artigos coletados até agora: 2530
Artigos coletados até agora: 2727
Artigos coletados até agora: 2920
Artigos coletados até agora: 3117
Artigos coletados até agora: 3315
Artigos coletados até agora: 3513
Artigos coletados até agora: 3703
Artigos coletados até agora: 3899
Artigos coletados até agora: 4097
Artigos coletados até agora: 4294
Artigos coletados até agora: 4493
Artigos coletados até agora: 4693
Artigos coletados até agora: 4893
Artigos coletados até agora: 5092
Artigos coletados até agora: 5286